<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c39-header.png" alt="Nextia Learning · Entity Resolution and Record Matching: Is This the Same Thing?" width="100%">

# Solution: MinHash and LSH

**[MinHash and LSH](https://learning.nextia-ai.com/courses/record-matching/m03/minhash-and-lsh/)** · Entity Resolution and Record Matching: Is This the Same Thing? · Module 3, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** cut titles into shingles, compute MinHash signatures and their agreement, and block two product feeds with MinHash LSH.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (about 2 MB). |
| **You should know** | Blocking keys and pair completeness, from [Blocking keys and sorted neighbourhood](https://learning.nextia-ai.com/courses/record-matching/m03/blocking-keys-and-sorted-neighbourhood/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (rapidfuzz 3.14.6, jellyfish 1.3.0, datasketch 2.0.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/record-matching/m03/minhash-and-lsh/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C39/M03-L03-minhash-and-lsh/minhash-and-lsh-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, rapidfuzz 3.14.6, jellyfish 1.3.0, datasketch 2.0.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2 rapidfuzz==3.14.6 jellyfish==1.3.0 datasketch==2.0.0
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'rapidfuzz', 'jellyfish', 'datasketch']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 2 MB) into a folder `larkfield-match` and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-match` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Every customer record and every product listing is **synthetic**: our code made them from invented people and products, and then added realistic errors on purpose (typing errors, nicknames, old addresses, shared family emails). Because the code knows the true person behind each record, you can always check an answer. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C39/data/dataset-card.md) explains the error model.

In [ ]:
# Setup: download the files into larkfield-match/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C39/"
FILES = {
    "data/products/feed_a.csv": "b5badaa6f0a813a19bb321263f46950100ef3a4e9a046163ed97e7af60fc94e2",
    "data/products/feed_b.csv": "c6bcc923db6b503859b20b5e4e1833d3376b08bd96b9d65c0392ae07a480e453",
    "data/products/truth.csv": "64adad1022e94e76841d3f34e339a0cc14cd1be809e2d69ab50a034d55b0b983",
    "data/truth.csv": "d951378129bdf30b649250e6d2127787e5630d66aa7cfe5793b1a110e37a06d3",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-match").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-match and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-match.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd` and matplotlib. It defines a few small helpers:

- `source(name)` reads one customer source as it is (`"shop"`, `"loyalty"` or `"garden"`); `records()` reads all three in shared columns (`record_id, source, name, email, phone, address, postcode, date_of_birth`), with the values exactly as the sources wrote them.
- `truth()` says which true customer (`entity_id`) each record belongs to, and its split (`train`, `validation` or `test`).
- `pair_measures(pairs)` measures a table of pairs (`record_id_l`, `record_id_r`) that a matcher calls "the same": pair precision, recall and F1 on the test split, as in the project's `match eval`.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, pair measures, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

FILES_OF = {"shop": "shop_accounts.csv", "loyalty": "loyalty_members.csv", "garden": "garden_centres.csv"}

def source(name):
    """One customer source as it is: its own columns, every value as text."""
    return pd.read_csv(f"data/sources/{FILES_OF[name]}", dtype=str, keep_default_na=False)

def records():
    """All customer records in shared columns, values as the sources wrote them."""
    s, l, g = source("shop"), source("loyalty"), source("garden")
    parts = [
        pd.DataFrame({"record_id": s.record_id, "source": "shop", "name": s.first_name + " " + s.last_name,
                      "email": s.email, "phone": s.phone, "address": s.address + ", " + s.city + " " + s.province,
                      "postcode": s.postcode, "date_of_birth": s.date_of_birth}),
        pd.DataFrame({"record_id": l.record_id, "source": "loyalty", "name": l.full_name, "email": l.email,
                      "phone": l.phone, "address": "", "postcode": l.postcode, "date_of_birth": l.date_of_birth}),
        pd.DataFrame({"record_id": g.record_id, "source": "garden", "name": g.name, "email": g.email,
                      "phone": g.phone, "address": g.address, "postcode": "", "date_of_birth": g.date_of_birth}),
    ]
    return pd.concat(parts, ignore_index=True)

def truth():
    """record_id -> entity_id (the true customer) and split."""
    return pd.read_csv("data/truth.csv", dtype=str)

def ordered(pairs):
    """The same pairs with the smaller record ID first, one row per pair."""
    lo = pairs[["record_id_l", "record_id_r"]].min(axis=1)
    hi = pairs[["record_id_l", "record_id_r"]].max(axis=1)
    out = pd.DataFrame({"record_id_l": lo, "record_id_r": hi})
    return out[out.record_id_l != out.record_id_r].drop_duplicates()

def true_pairs():
    """Every pair of records of the same true customer, with its split."""
    t = truth()
    m = t.merge(t, on="entity_id", suffixes=("_l", "_r"))
    m = m[m.record_id_l < m.record_id_r]
    return m[["record_id_l", "record_id_r", "split_l"]].rename(columns={"split_l": "split"})

RANK = {"train": 0, "validation": 1, "test": 2}

def pair_measures(pairs, split="test"):
    """Pair precision, recall and F1 of the pairs a matcher calls the same (a pair is in the test split when one
    of its records belongs to a test customer)."""
    p = ordered(pairs)
    t = truth().set_index("record_id")
    rank = t.split.map(RANK)
    p = p.assign(split=np.maximum(p.record_id_l.map(rank), p.record_id_r.map(rank)).map({v: k for k, v in RANK.items()}),
                 same=(p.record_id_l.map(t.entity_id) == p.record_id_r.map(t.entity_id)))
    tp_all = true_pairs()
    p, tp_all = p[p.split == split], tp_all[tp_all.split == split]
    tp = int(p.same.sum())
    precision = tp / len(p) if len(p) else 0.0
    recall = tp / len(tp_all) if len(tp_all) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3),
            "pairs": len(p), "right": tp, "true_pairs": len(tp_all)}

def same_value_pairs(df, col):
    """Every pair of records that share a non-empty value of col."""
    d = df[df[col].notna() & (df[col] != "")][["record_id", col]]
    sizes = d[col].value_counts()
    n_pairs = int((sizes * (sizes - 1) // 2).sum())
    if n_pairs > 20_000_000:   # protect the computer's memory from a key that is too wide
        raise ValueError(f"This key makes {n_pairs:,} pairs: too many to hold in memory. Use a narrower key.")
    m = d.merge(d, on=col, suffixes=("_l", "_r"))
    return m[m.record_id_l < m.record_id_r][["record_id_l", "record_id_r"]].drop_duplicates()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Five titles, cut into shingles

Two titles for one hose seldom agree exactly. Cut each title into **shingles**: every run of 3 characters. Run the next cell.

In [ ]:
# five titles of the two feeds, without punctuation: NF-03673, CL-03459, NF-02877, NF-01651, CL-00741
titles = ["Mossway Vinyl Garden Hose 30 ft Brown", "gdn hose 9.1 m vinyl brown MOSSWAY",
          "Mossway Vinyl Garden Hose 30 ft Brown Pack of 2", "Copperwren Ceramic Planter 12 in Black",
          "planter 30.5 cm ceramic black Copperwren"]

def shingles(text, k=3):
    """The set of k-character pieces of a text, after lower case."""
    t = " ".join(str(text).lower().split())
    return {t[i:i + k] for i in range(max(1, len(t) - k + 1))}

def jaccard(a, b):
    return len(a & b) / len(a | b)

sh = [shingles(t) for t in titles]
print(sorted(sh[0])[:12], "...", len(sh[0]), "shingles")
print("A-B:", len(sh[0] & sh[1]), "shared,", len(sh[0] | sh[1]), "in all")

The **Jaccard similarity** of two sets is the number of shared items divided by the number of items in either set.

> **Predict.** Which pair of the five titles is the most similar? (A and B are one hose, C is a pack of two hoses, D and E are one planter.)

In [ ]:
names = "ABCDE"
table = pd.DataFrame([{"pair": names[i] + "-" + names[j], "jaccard": round(jaccard(sh[i], sh[j]), 3)}
                      for i in range(5) for j in range(i + 1, 5)])
print(table.to_string(index=False))

## 2. MinHash signatures

For each of 100 hash functions, a MinHash signature keeps the **smallest** hash of the title's shingles. Two titles have the same smallest hash with a chance equal to their Jaccard similarity. Run the next cell: it makes the signature of each title from that title alone.

In [ ]:
import zlib
rng = np.random.default_rng(1)
N = 100
A = rng.integers(1, 2**31 - 1, size=N, dtype=np.int64)
B = rng.integers(0, 2**31 - 1, size=N, dtype=np.int64)

def signature(shingle_set):
    """N MinHash values: for each of N hash functions, the smallest hash of the set."""
    h = np.array([zlib.crc32(s.encode()) for s in shingle_set], dtype=np.int64)
    return ((np.outer(A, h) + B[:, None]) % (2**31 - 1)).min(axis=1)

sig = [signature(s) for s in sh]
print("A:", sig[0][:5], "\nB:", sig[1][:5])

## 3. Your turn: the agreement of two signatures

Compute the share of the 100 positions where the signatures of A and B are equal.

In [ ]:
agree_ab = (sig[0] == sig[1]).mean()
print("A and B agree on", agree_ab, "of the values; their Jaccard is", round(jaccard(sh[0], sh[1]), 3))

In [ ]:
expect('agree_ab', agree_ab, 0.4)

Run the next cell: the agreement of every pair, next to its Jaccard similarity.

In [ ]:
if agree_ab is not ...:
    table["minhash"] = [round(float((sig[i] == sig[j]).mean()), 2) for i in range(5) for j in range(i + 1, 5)]
    print(table.to_string(index=False))

## 4. Bands make an S-curve

LSH cuts each signature into bands. A pair becomes a candidate when **all** values of **any one** band are equal. With *b* bands of *r* values, a pair with Jaccard *s* becomes a candidate with chance 1 − (1 − *s*^*r*)^*b*.

In [ ]:
s = np.linspace(0, 1, 101)
bands, rows = 25, 5
plt.plot(s, 1 - (1 - s**rows) ** bands, label="25 bands of 5")
plt.plot(s, s**rows, "--", label="one band of 5")
plt.xlabel("Jaccard similarity"); plt.ylabel("chance to become a candidate"); plt.legend(); plt.show()

## 5. Your turn: one point of the curve

Compute the chance for a pair with Jaccard 0.6, with 25 bands of 5 values.

In [ ]:
p_06 = 1 - (1 - 0.6**5) ** 25
print(round(p_06, 3))

In [ ]:
expect('p_06', None if p_06 is ... else round(p_06, 3), 0.868)

## 6. MinHash LSH on the two supplier feeds

The `datasketch` library does the three steps for us. The next cell reads the two feeds and finds the true pairs across them.

In [ ]:
a = pd.read_csv("data/products/feed_a.csv", dtype=str, keep_default_na=False)
b = pd.read_csv("data/products/feed_b.csv", dtype=str, keep_default_na=False)
products = pd.concat([
    pd.DataFrame({"product_id": a.product_id, "text": (a.brand + " " + a.title).str.lower()}),
    pd.DataFrame({"product_id": b.product_id, "text": (b.supplier_brand + " " + b.title).str.lower()}),
], ignore_index=True)
ptruth = pd.read_csv("data/products/truth.csv", dtype=str)
m = ptruth.merge(ptruth, on="product_entity", suffixes=("_l", "_r"))
ptrue = m[(m.product_id_l.str[:2] == "NF") & (m.product_id_r.str[:2] == "CL")]
ptrue = ptrue.rename(columns={"product_id_l": "a", "product_id_r": "b"})[["a", "b"]]
possible = len(a) * len(b)
print(f"{len(a):,} x {len(b):,} = {possible:,} possible pairs; {len(ptrue):,} true pairs")

> **Predict.** With threshold 0.5, how many of the true pairs will become candidates?

In [ ]:
import time
from datasketch import MinHash, MinHashLSH

def lsh_candidates(threshold):
    lsh = MinHashLSH(threshold=threshold, num_perm=128)
    sigs = {}
    for pid, text in zip(products.product_id, products.text):
        mh = MinHash(num_perm=128, seed=1)
        mh.update_batch([x.encode() for x in shingles(text)])
        sigs[pid] = mh
        lsh.insert(pid, mh)
    rows = {(p, q) for p, mh in sigs.items() if p.startswith("NF") for q in lsh.query(mh) if q.startswith("CL")}
    return pd.DataFrame(sorted(rows), columns=["a", "b"])

t0 = time.perf_counter()
cand = lsh_candidates(0.5)
kept = ptrue.merge(cand, on=["a", "b"])
lsh_result = {"candidates": len(cand), "pair_completeness": round(len(kept) / len(ptrue), 3),
              "reduction_ratio": round(1 - len(cand) / possible, 4)}
print(lsh_result, f"in {time.perf_counter() - t0:.1f} s")

> **Check.** MinHash LSH makes 33,470 candidate pairs and keeps 0.73 of the true pairs. This notebook reads the raw title in lower case. The project reads clean text (the brand and the title words, without size and pack), so its numbers differ.

Try `lsh_candidates(0.3)` and `lsh_candidates(0.8)`, and measure each one. What happens to the true pairs?

## 7. In the project

```sh
match block --blocking p-brand
match block --blocking p-minhash
match block --blocking p-minhash+gtin
```

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [What blocking loses](https://learning.nextia-ai.com/courses/record-matching/m03/what-blocking-loses/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/record-matching/m03/minhash-and-lsh/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every customer record and every product listing here is synthetic: our code made them from invented people and products. No real person is in this data.